In [1]:
import pandas as pd
import plotly.express as px
from pathlib import Path

RAW = Path("../data/raw")
file = next(RAW.glob("*kldb2010-zeitreihe-d*.xlsx"))

raw = pd.read_excel(file, sheet_name="1", header=None, skiprows=9).iloc[:, :40]
names = pd.read_excel(file, sheet_name="1", header=None, nrows=8).iloc[7, 3:40].tolist()
names = [" ".join(str(n).replace("-", "").split()) for n in names]
raw.columns = ["year", "quarter", "total"] + names
raw["year"] = pd.to_numeric(raw["year"].astype(str).str.strip(), errors="coerce").ffill()
raw["quarter"] = raw["quarter"].astype(str).str.strip()

june = raw[raw["quarter"] == "Juni"].set_index("year").drop(columns="quarter").astype(float)
growth = pd.DataFrame({"jobs_2015": june.loc[2015], "jobs_2025": june.loc[2025]})
growth["change_pct"] = ((growth["jobs_2025"] / growth["jobs_2015"] - 1) * 100).round(1)
avg_growth = growth.loc["total", "change_pct"]
growth = growth.drop(index=["total", "Keine Zuordnung möglich"])
growth["group_code"] = growth.index.str[:2]      # "43 Informatik…" → "43"
growth["group"] = growth.index.str[3:]
growth.head()

,jobs_2015,jobs_2025,change_pct,group_code,group
"11 Land, Tier, Forstwirtschaftsberufe",229790.0,247256.0,7.6,11,"Land, Tier, Forstwirtschaftsberufe"
12 Gartenbauberufe und Floristik,258583.0,279359.0,8.0,12,Gartenbauberufe und Floristik
"21 Rohstoffgewinn und aufbereitung, Glas und Keramikherstellung und verarbeitung",127724.0,109334.0,-14.4,21,"Rohstoffgewinn und aufbereitung, Glas und Kera..."
"22 Kunststoffherstellung und verarbeitung, Holzbe und verarbeitung",533794.0,477009.0,-10.6,22,"Kunststoffherstellung und verarbeitung, Holzbe..."
"23 Papier und Druckberufe, technische Mediengestaltung",277360.0,241478.0,-12.9,23,"Papier und Druckberufe, technische Mediengesta..."


In [3]:
epa = pd.read_csv(RAW / "epa_2025.csv", sep=";", decimal=",", encoding="utf-8-sig",
                  na_values=["X", " "], dtype={"Beruf.BA_SCHL": str})
epa25 = epa[(epa["Jahr.DESC"] == 2025)
            & (epa["Bundesland RZ.BA_SCHL"] == "d")
            & (epa["Beruf.BA_SCHL"].str.len() == 4)].copy()
epa25["group_code"] = epa25["Beruf.BA_SCHL"].str[:2]

shortage = epa25.groupby("group_code").agg(
    rated=("Einstufung", "size"),
    shortage=("Einstufung", lambda x: (x == "Engpassberuf").sum()))
shortage["shortage_share"] = (shortage["shortage"] / shortage["rated"] * 100).round(0)

matrix = growth.merge(shortage, left_on="group_code", right_index=True)
avg_short = round(shortage["shortage"].sum() / shortage["rated"].sum() * 100)
print("Среднее: рост", avg_growth, "%, доля дефицита", avg_short, "%")
matrix.sort_values("shortage_share", ascending=False)

Среднее: рост 13.4 %, доля дефицита 30 %


,jobs_2015,jobs_2025,change_pct,group_code,group,rated,shortage,shortage_share
"21 Rohstoffgewinn und aufbereitung, Glas und Keramikherstellung und verarbeitung",127724.0,109334.0,-14.4,21,"Rohstoffgewinn und aufbereitung, Glas und Kera...",2,2,100.0
29 Lebensmittelherstellung und verarbeitung,806672.0,868399.0,7.7,29,Lebensmittelherstellung und verarbeitung,9,7,78.0
32 Hoch und Tiefbauberufe,553493.0,595694.0,7.6,32,Hoch und Tiefbauberufe,16,11,69.0
34 Gebäude und versorgungstechnische Berufe,674348.0,793432.0,17.7,34,Gebäude und versorgungstechnische Berufe,17,11,65.0
81 Medizinische Gesundheitsberufe,2347031.0,2923892.0,24.6,81,Medizinische Gesundheitsberufe,38,22,58.0
"22 Kunststoffherstellung und verarbeitung, Holzbe und verarbeitung",533794.0,477009.0,-10.6,22,"Kunststoffherstellung und verarbeitung, Holzbe...",7,4,57.0
62 Verkaufsberufe,2056653.0,2061079.0,0.2,62,Verkaufsberufe,23,11,48.0
"82 Nicht medizinische Gesundheits, Körperpflege und Wellnessberufe, Medizintechnik",832780.0,978128.0,17.5,82,"Nicht medizinische Gesundheits, Körperpflege u...",17,8,47.0
"26 Mechatronik, Energie und Elektroberufe",993360.0,1077095.0,8.4,26,"Mechatronik, Energie und Elektroberufe",25,11,44.0
12 Gartenbauberufe und Floristik,258583.0,279359.0,8.0,12,Gartenbauberufe und Floristik,9,4,44.0


In [4]:
fig = px.scatter(matrix, x="change_pct", y="shortage_share", size="jobs_2025", text="group_code",
                 hover_name="group", hover_data={"jobs_2025": ":,", "rated": True, "group_code": False},
                 size_max=45, height=700,
                 title="Growing, shrinking, short of workers: 36 occupation groups in Germany",
                 labels={"change_pct": "Employment change 2015–2025, %",
                         "shortage_share": "Share of shortage occupations 2025, %"})
fig.add_vline(x=avg_growth, line_dash="dash")
fig.add_hline(y=avg_short, line_dash="dash")
for x, y, t in [(45, 95, "Growing & short"), (-15, 95, "Shrinking & still short"),
                (45, -5, "Growing, no shortage"), (-15, -5, "Shrinking, no shortage")]:
    fig.add_annotation(x=x, y=y, text=f"<b>{t}</b>", showarrow=False, opacity=0.6)
fig.update_traces(textposition="middle center")
fig.show()
fig.write_html("../output/charts/03_matrix_growth_shortage.html")

## Finding 3: Growth and shortage do not always go together

- **Growing & short:** healthcare (81: +24.6%, 58% of occupations in shortage), building services (34), education & social work (83: +43.8%).
- **Growing, no shortage:** IT (43: +67.2%, only 7% in shortage), management (71), marketing (92), humanities (91).
- **Shrinking & still short:** metalworking (24: −17.2%, 31%), plastics/wood (22: −10.6%, 57%).
- Hypothesis for discussion: in shrinking occupations workers leave (retirement, career change) faster than jobs disappear.

Sources: BA Engpassanalyse 2025; BA Beschäftigte nach Berufen, June 2015 vs June 2025. Own calculations.
